In [ ]:
from influxdb_client import InfluxDBClient
from influxdb_client.client.write_api import SYNCHRONOUS
import pandas as pd


def load_data_bucket(bucket: str) -> pd.DataFrame:
    # Replace these with your InfluxDB token, organization, and bucket
    org = "ur3e"
    token = "qFEMOoL0ghl90ghWnzq61MNfoI20_cZrAffwooMdgUl8tk-eIUVsI4Vv7vCsLR9eCeLPPWGYNemWblSlzv3WSw=="

    # Initialize the client
    client = InfluxDBClient(url="http://localhost:8086", token=token, org=org)
    write_api = client.write_api(write_options=SYNCHRONOUS)   
    query_api = client.query_api()

    time_start = "-1000h"
    time_stop = "now()"

    # Get sensor data
    query_sensor_data = f'''
        from(bucket: "{bucket}")
        |> range(start: {time_start}, stop: {time_stop})
        |> filter(fn: (r) => r["_measurement"] == "robotarm.pt.state")
        |> filter(fn: (r) => r["_field"] =~ /(((q_actual|qd_actual|q_target)_[0-6])|(tcp_pose_[0-6]))/)
        |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
        '''

    df_sensor_data = query_api.query_data_frame(query_sensor_data)

    # Get wear injections
    query_wear_injections = f'''
        from(bucket: "{bucket}")
        |> range(start: {time_start}, stop: {time_stop})
        |> filter(fn: (r) => r["_measurement"] == "robotarm.ctrl")
        |> filter(fn: (r) => r["_field"] =~ /joints_0/)
        |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
        '''

    df_wear_injections = query_api.query_data_frame(query_wear_injections)

    # Remove unwanted columns
    cols_to_exclude = ["result", "table", "_start", "_stop", "_measurement", "msg_type"]
    df_filtered_sensor_data = df_sensor_data.drop(columns=cols_to_exclude)
    df_filtered_wear_injections = df_wear_injections.drop(columns=["result", "table", "_start", "_stop", "_measurement", "msg_type"])

    # Construct labeled dataframe
    # 1. Ensure time columns are datetime objects 
    df_filtered_wear_injections['_time'] = pd.to_datetime(df_filtered_wear_injections['_time'])
    df_filtered_sensor_data['_time'] = pd.to_datetime(df_filtered_sensor_data['_time'])

    # 2. Extract both the timestamps AND the joint numbers into lists
    reference_timestamps = df_filtered_wear_injections['_time'].tolist()
    joint_numbers = df_filtered_wear_injections['joints_0'].tolist()

    # 3. Reorder in terms of joint_numbers
    # Zip pairs them up: (joint, time), sorts by the first element (joint), then unzips
    sorted_pairs = sorted(zip(joint_numbers, reference_timestamps))
    joint_numbers, reference_timestamps = map(list, zip(*sorted_pairs))

    # 4. Use zip() to loop through both lists at the same time
    for ref_time, joint_num in zip(reference_timestamps, joint_numbers):
        # Name the column using the exact joint number (e.g., after_injection_1)
        col_name = f'joint_{joint_num}_worn'
        
        # Apply the binary comparison logic
        df_filtered_sensor_data[col_name] = (df_filtered_sensor_data['_time'] > ref_time).astype(int)

    # Check your newly added columns
    return df_filtered_sensor_data

In [ ]:
# Put all dataframes from InfluxDB into lists
train_dfs = []
for i in range(1, 7):
    train_dfs.append(load_data_bucket(f"train_data_{i}"))

# Remove timestamps for all dataframes
for i, df in enumerate(train_dfs):
    train_dfs[i] = df.drop(columns=["_time"])

test_dfs = []
for i in range(1, 7):
    test_dfs.append(load_data_bucket(f"test_data_{i}"))

# --- Saving DataFrames ---

# 1. Define the directory and create it if it doesn't exist
output_dir = Path("./data")
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Save the Training sets
for i, df in enumerate(train_dfs):
    # Creates a path like: ./data/train_scaled_0.csv
    file_path = output_dir / f"train_{i}.csv" 
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")

# 3. Save the Testing sets
for i, df in enumerate(test_dfs):
    # Creates a path like: ./data/test_scaled_0.csv
    file_path = output_dir / f"test_{i}.csv"
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Fit scaler ONLY on training data
scaler = StandardScaler()
train_data_combined = pd.concat(train_dfs) # Concat temporarily just to fit the scaler
scaler.fit(train_data_combined)

# Scale all DataFrames independently (Pre-processing)
train_dfs_scaled = [pd.DataFrame(scaler.transform(df), columns=df.columns) for df in train_dfs]
test_dfs_scaled = [pd.DataFrame(scaler.transform(df), columns=df.columns) for df in test_dfs]

In [ ]:
train_data_combined.columns[-6:]

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class MultiSeriesDataset(Dataset):
    def __init__(self, dataframes, seq_length):
        """
        Args:
            dataframes (list of pd.DataFrame): List of independent time-series dataframes.
            seq_length (int): How many past time steps to use as input (X).
        """
        self.seq_length = seq_length
        
        self.X = []
        self.y = []
        
        # Process each dataframe individually to prevent cross-contamination
        for df in dataframes:
            self._process_dataframe(df.values)
            
        # Convert the accumulated lists into PyTorch tensors
        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def _process_dataframe(self, data):
        # We need at least seq_length + 1 rows to make one prediction
        if len(data) <= self.seq_length:
            return 
            
        # Create sliding windows
        for i in range(len(data) - self.seq_length):
            # X: The sequence of past observations
            # If you want to EXCLUDE the targets from your inputs, use: data[i : i + self.seq_length, :-6]
            window_x = data[i : i + self.seq_length, :-6]
            
            # y: The target values at the next time step (the last 6 columns)
            window_y = data[i + self.seq_length, -6:]
            
            self.X.append(window_x)
            self.y.append(window_y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_ds = MultiSeriesDataset(train_dfs, 100)
test_ds = MultiSeriesDataset(test_dfs, 100)